# LR Multiclass Evaluation

Synthetic IPv6 multiclass Logistic Regression with five saved seed runs and official split files.


In [ ]:
import json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

ROOT = "/kaggle/input/datasets/synthetic-data/synthetic_ipv6_grounded_v3_32x32"
OUT_DIR = Path("results"); OUT_DIR.mkdir(exist_ok=True, parents=True)

DROP_COLS = ["record_id","window_start_utc","window_end_utc","src_ip","dst_ip"]
TARGET = "label"
SEEDS = [11, 22, 33, 44, 55]


In [6]:
flows = pd.read_csv(f"{ROOT}/flows.csv")
train_split = pd.read_csv(f"{ROOT}/train.csv")
val_split   = pd.read_csv(f"{ROOT}/val.csv")
test_split  = pd.read_csv(f"{ROOT}/test.csv")

train = flows.merge(train_split[["record_id"]], on="record_id", how="inner")
val   = flows.merge(val_split[["record_id"]],   on="record_id", how="inner")
test  = flows.merge(test_split[["record_id"]],  on="record_id", how="inner")

print("Rows:", {"train": len(train), "val": len(val), "test": len(test)})
display(flows["label"].value_counts())


Rows: {'train': 2100, 'val': 450, 'test': 450}


label
Benign            2829
Exploits            50
Fuzzers             44
Generic             33
Reconnaissance      16
DoS                 10
Shellcode            8
Backdoor             5
Analysis             3
Worms                2
Name: count, dtype: int64

In [7]:
def split_xy(df):
    X = df.drop(columns=[TARGET] + DROP_COLS, errors="ignore")
    y = df[TARGET].astype(str).values
    return X, y

X_train, y_train = split_xy(train)
X_test, y_test = split_xy(test)

cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]
print("Numeric cols:", len(num_cols), "| Categorical cols:", cat_cols)

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler(with_mean=False))]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
], remainder="drop")


Numeric cols: 19 | Categorical cols: ['transport', 'app_proto', 'direction']


In [8]:
runs = []
for seed in SEEDS:
    clf = LogisticRegression(max_iter=5000, solver="saga", class_weight="balanced", random_state=seed)
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_test, pred, average="weighted", zero_division=0)

    runs.append({
        "seed": seed,
        "accuracy": acc,
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f1_macro,
        "weighted_f1": f1_w
    })

runs_df = pd.DataFrame(runs)
display(runs_df)

summary = {
    "seeds": SEEDS,
    "macro_f1_mean": float(runs_df["macro_f1"].mean()),
    "macro_f1_std": float(runs_df["macro_f1"].std(ddof=1)),
}
print("Macro-F1 (mean ± std):", summary["macro_f1_mean"], "±", summary["macro_f1_std"])


,seed,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,11,0.924444,0.360922,0.475546,0.398047,0.94166
1,22,0.924444,0.360922,0.475546,0.398047,0.94166
2,33,0.924444,0.360922,0.475546,0.398047,0.94166
3,44,0.924444,0.360922,0.475546,0.398047,0.94166
4,55,0.924444,0.360922,0.475546,0.398047,0.94166


Macro-F1 (mean ± std): 0.3980469289164941 ± 0.0


In [9]:
all_classes = sorted(flows[TARGET].unique().tolist())
support = pd.Series({c:int((y_test==c).sum()) for c in all_classes}).sort_values()
support_df = support.reset_index()
support_df.columns=["class","test_support"]
support_df["unstable"] = support_df["test_support"] < 5
display(support_df)


,class,test_support,unstable
0,Backdoor,0,True
1,Analysis,1,True
2,DoS,1,True
3,Shellcode,1,True
4,Worms,1,True
5,Reconnaissance,3,True
6,Generic,5,False
7,Fuzzers,6,False
8,Exploits,7,False
9,Benign,425,False


In [10]:
runs_df.to_csv(OUT_DIR/"lr_multiclass_5seed_runs.csv", index=False)
(OUT_DIR/"lr_multiclass_5seed_summary.json").write_text(json.dumps({
    "seeds": SEEDS,
    "macro_f1_mean": float(runs_df["macro_f1"].mean()),
    "macro_f1_std": float(runs_df["macro_f1"].std(ddof=1)),
    "accuracy_mean": float(runs_df["accuracy"].mean()),
    "weighted_f1_mean": float(runs_df["weighted_f1"].mean()),
}, indent=2))
support_df.to_csv(OUT_DIR/"test_support_unstable_flags.csv", index=False)

clf = LogisticRegression(max_iter=5000, solver="saga", class_weight="balanced", random_state=SEEDS[0])
pipe = Pipeline([("prep", preprocess), ("clf", clf)])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

(OUT_DIR/"lr_multiclass_classification_report.txt").write_text(
    classification_report(y_test, pred, zero_division=0, digits=4)
)
cm = confusion_matrix(y_test, pred, labels=all_classes)
pd.DataFrame(cm, index=all_classes, columns=all_classes).to_csv(OUT_DIR/"lr_multiclass_confusion_matrix.csv")

print("Saved results to:", OUT_DIR.resolve())


Saved results to: /kaggle/working/results
